# Algorithmic Trading Strategy Backtest

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import scipy.stats as stats
import pytz
import yfinance as yf
from datetime import time
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import itertools
from IPython.display import display

## 2. Strategy & Helper Functions

In [ ]:
def fetch_data(ticker, period='2y', interval='1h'):
    """Fetches and preprocesses historical data from Yahoo Finance."""
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True, progress=False)
    if df.empty:
        print(f"No data found for ticker {ticker}. Skipping.")
        return None, None
    df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    if df.index.tz is None: df = df.tz_localize('UTC')
    else: df = df.tz_convert('UTC')
    ny_tz = pytz.timezone('America/New_York')
    df = df.tz_convert(ny_tz)
    df.dropna(inplace=True)
    ohlc_dict = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}
    df_4h = df.resample('4H').apply(ohlc_dict).dropna()
    return df, df_4h

def check_market_regime(current_timestamp, df_4h, params):
    last_4h_data = df_4h.asof(current_timestamp)
    if pd.isna(last_4h_data.any()): return "SIDEWAYS_RANGE"
    fast_sma = last_4h_data[f"SMA_{params['Period_Fast_SMA']}"]
    medium_sma = last_4h_data[f"SMA_{params['Period_Medium_SMA']}"]
    slow_sma = last_4h_data[f"SMA_{params['Period_Slow_SMA']}"]
    adx = last_4h_data[f"ADX_{params['ATR_Period']}"]
    if fast_sma > medium_sma > slow_sma and adx > params['ADX_Threshold']: return "BULLISH_TREND"
    if fast_sma < medium_sma < slow_sma and adx > params['ADX_Threshold']: return "BEARISH_TREND"
    return "SIDEWAYS_RANGE"

def check_time_filters(current_timestamp):
    current_time = current_timestamp.time()
    can_open_new = (time(8, 0) <= current_time < time(15, 0))
    is_active_session = (time(8, 0) <= current_time < time(16, 0))
    return {'can_open_new': can_open_new, 'is_active_session': is_active_session}

def find_swing_points(series, order=5):
    high_peaks_indices, _ = find_peaks(series, distance=order, width=1)
    low_peaks_indices, _ = find_peaks(-series, distance=order, width=1)
    return high_peaks_indices, low_peaks_indices

def detect_break_of_structure(df_slice, current_index, direction, params, atr_multiplier=1.0):
    current_candle = df_slice.loc[current_index]
    atr = current_candle[f'ATRr_{params["ATR_Period"]}']
    if direction == "Bullish":
        swing_highs = df_slice.loc[:current_index][:-1]
        last_swing_high = swing_highs[swing_highs['is_swing_high']]
        if not last_swing_high.empty:
            last_swing_high_price = last_swing_high.iloc[-1]['High']
            if current_candle['Close'] > last_swing_high_price + (atr * atr_multiplier):
                return True, last_swing_high.index[-1]
    elif direction == "Bearish":
        swing_lows = df_slice.loc[:current_index][:-1]
        last_swing_low = swing_lows[swing_lows['is_swing_low']]
        if not last_swing_low.empty:
            last_swing_low_price = last_swing_low.iloc[-1]['Low']
            if current_candle['Close'] < last_swing_low_price - (atr * atr_multiplier):
                return True, last_swing_low.index[-1]
    return False, None

def detect_fair_value_gap(df_slice, current_index, direction):
    if len(df_slice.loc[:current_index]) < 3: return False, None
    three_candles = df_slice.loc[:current_index].iloc[-3:]
    if direction == "Bullish" and three_candles.iloc[0]['High'] < three_candles.iloc[2]['Low']:
        return True, (three_candles.iloc[0]['High'], three_candles.iloc[2]['Low'])
    if direction == "Bearish" and three_candles.iloc[0]['Low'] > three_candles.iloc[2]['High']:
        return True, (three_candles.iloc[2]['High'], three_candles.iloc[0]['Low'])
    return False, None

def detect_order_block(df_slice, bos_timestamp, direction, params, body_atr_multiplier=1.0):
    search_slice = df_slice.loc[:bos_timestamp]
    if direction == "Bullish":
        potential_obs = search_slice[search_slice['Close'] < search_slice['Open']]
        if not potential_obs.empty:
            last_down_candle = potential_obs.iloc[-1]
            atr = last_down_candle[f'ATRr_{params["ATR_Period"]}']
            if abs(last_down_candle['Open'] - last_down_candle['Close']) > atr * body_atr_multiplier:
                return True, (last_down_candle['Low'], last_down_candle['High'])
    elif direction == "Bearish":
        potential_obs = search_slice[search_slice['Close'] > search_slice['Open']]
        if not potential_obs.empty:
            last_up_candle = potential_obs.iloc[-1]
            atr = last_up_candle[f'ATRr_{params["ATR_Period"]}']
            if abs(last_up_candle['Open'] - last_up_candle['Close']) > atr * body_atr_multiplier:
                return True, (last_up_candle['Low'], last_up_candle['High'])
    return False, None

def find_trade_setup(df_slice, current_index, direction, params):
    current_candle = df_slice.loc[current_index]
    if (direction == "Bullish" and current_candle['Close'] < current_candle['VWAP_D']) or \
       (direction == "Bearish" and current_candle['Close'] > current_candle['VWAP_D']):
        return None
    bos_detected, bos_timestamp = detect_break_of_structure(df_slice, current_index, direction, params)
    if not bos_detected: return None
    leg_slice = df_slice.loc[bos_timestamp:current_index]
    fvg_detected, fvg_zone = detect_fair_value_gap(leg_slice, current_index, direction)
    ob_detected, ob_zone = detect_order_block(leg_slice, current_index, direction, params)
    if not fvg_detected and not ob_detected: return None
    entry_zone = ob_zone if ob_detected else fvg_zone
    sl_level = entry_zone[0] if direction == 'Bullish' else entry_zone[1]
    return {'direction': direction, 'entry_zone': entry_zone, 'stop_loss_level': sl_level, 'detected_at': current_index}

def calculate_position_size(equity, risk_pct, entry, sl, price_per_point=1):
    if entry == sl: return 0
    dollar_risk = equity * (risk_pct / 100)
    sl_dist = abs(entry - sl) * price_per_point
    return dollar_risk / sl_dist if sl_dist > 0 else 0

def calculate_exit_levels(entry, sl, direction, rr_ratio=2.0):
    sl_dist = abs(entry - sl)
    tp_dist = sl_dist * rr_ratio
    tp = entry + tp_dist if direction == "Bullish" else entry - tp_dist
    return {'stop_loss': sl, 'profit_target': tp}

## 3. Backtester Class

In [ ]:
class Backtester:
    def __init__(self, df, df_4h, strategy_params, initial_equity=100000.0, risk_percent=1.0, commission=4.95, slippage_ticks=1):
        self.df, self.df_4h, self.params = df, df_4h, strategy_params
        self.initial_equity, self.risk_percent, self.commission, self.slippage_ticks = initial_equity, risk_percent, commission, slippage_ticks
        self.equity, self.daily_high_equity = initial_equity, initial_equity
        self.equity_curve, self.trade_history, self.active_setups = [], [], []
        self.in_trade, self.kill_switch_today, self.current_day, self.open_trade = False, False, None, {}

    def _get_tick_size(self): return 0.0001 # Forex standard

    def _open_trade(self, ts, price, direction, sl):
        slippage = self.slippage_ticks * self._get_tick_size()
        entry_price = price + slippage if direction == "Bullish" else price - slippage
        size = calculate_position_size(self.equity, self.risk_percent, entry_price, sl)
        if size <= 0: return
        exits = calculate_exit_levels(entry_price, sl, direction)
        self.equity -= self.commission
        self.in_trade = True
        self.open_trade = {'entry_timestamp': ts, 'entry_price': entry_price, 'direction': direction, 'position_size': size, **exits, 'pnl': 0}
        self.trade_history.append(self.open_trade.copy())

    def _close_trade(self, ts, price, reason):
        slippage = self.slippage_ticks * self._get_tick_size()
        exit_price = price - slippage if self.open_trade['direction'] == "Bullish" else price + slippage
        pnl = (exit_price - self.open_trade['entry_price']) * self.open_trade['position_size'] * (1 if self.open_trade['direction'] == "Bullish" else -1)
        self.equity += pnl - self.commission
        self.trade_history[-1].update({'exit_timestamp': ts, 'exit_price': exit_price, 'pnl': pnl, 'exit_reason': reason})
        self.in_trade, self.open_trade = False, {}

    def run(self):
        for index, row in self.df.iterrows():
            if self.current_day != index.date():
                self.current_day, self.kill_switch_today = index.date(), False
                self.daily_high_equity = self.equity
            self.daily_high_equity = max(self.daily_high_equity, self.equity)
            drawdown = (self.daily_high_equity - self.equity) / self.daily_high_equity if self.daily_high_equity > 0 else 0
            if drawdown > (self.params.get('MaxDailyDrawdown_Percentage', 3.0) / 100):
                self.kill_switch_today = True
            self.equity_curve.append({'timestamp': index, 'equity': self.equity})
            if self.kill_switch_today:
                if self.in_trade: self._close_trade(index, row['Close'], "Max Daily Drawdown")
                continue
            time_filters = check_time_filters(index)
            if self.in_trade:
                if not time_filters['is_active_session']: self._close_trade(index, row['Close'], "End of Session")
                elif self.open_trade['direction'] == "Bullish" and row['Low'] <= self.open_trade['stop_loss']: self._close_trade(index, self.open_trade['stop_loss'], "Stop Loss Hit")
                elif self.open_trade['direction'] == "Bullish" and row['High'] >= self.open_trade['profit_target']: self._close_trade(index, self.open_trade['profit_target'], "Profit Target Hit")
                elif self.open_trade['direction'] == "Bearish" and row['High'] >= self.open_trade['stop_loss']: self._close_trade(index, self.open_trade['stop_loss'], "Stop Loss Hit")
                elif self.open_trade['direction'] == "Bearish" and row['Low'] <= self.open_trade['profit_target']: self._close_trade(index, self.open_trade['profit_target'], "Profit Target Hit")
                continue
            if time_filters['can_open_new']:
                entry_low, entry_high = 0, 0
                for setup in self.active_setups[:]:
                    entry_low, entry_high = setup['entry_zone']
                    if setup['direction'] == 'Bullish' and row['Low'] <= entry_high:
                        self._open_trade(index, entry_high, 'Bullish', setup['stop_loss_level'])
                        self.active_setups = []
                        break
                    elif setup['direction'] == 'Bearish' and row['High'] >= entry_low:
                        self._open_trade(index, entry_low, 'Bearish', setup['stop_loss_level'])
                        self.active_setups = []
                        break
                if self.in_trade: continue
                market_regime = check_market_regime(index, self.df_4h, self.params)
                if market_regime != 'SIDEWAYS_RANGE':
                    direction = 'Bullish' if market_regime == 'BULLISH_TREND' else 'Bearish'
                    new_setup = find_trade_setup(self.df.loc[:index], index, direction, self.params)
                    if new_setup: self.active_setups.append(new_setup)
        return pd.DataFrame(self.trade_history)

## 4. Performance & Analysis Functions

In [ ]:
def calculate_performance_metrics(trade_log, equity_curve):
    if trade_log.empty: return { 'error': 'No trades to analyze.' }
    metrics = {}
    gross_profit = trade_log[trade_log['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(trade_log[trade_log['pnl'] < 0]['pnl'].sum())
    metrics['Profit Factor'] = gross_profit / gross_loss if gross_loss != 0 else float('inf')
    metrics['Win Rate (%)'] = (trade_log['pnl'] > 0).mean() * 100
    metrics['Total Trades'] = len(trade_log)
    eq_df = pd.DataFrame(equity_curve).set_index('timestamp')
    eq_df['returns'] = eq_df['equity'].pct_change().fillna(0)
    eq_df['drawdown'] = (eq_df['equity'] - eq_df['equity'].cummax()) / eq_df['equity'].cummax()
    metrics['Max Drawdown (%)'] = eq_df['drawdown'].min() * 100
    daily_returns = eq_df['returns'].resample('D').sum()
    metrics['Sharpe Ratio'] = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252) if daily_returns.std() != 0 else 0
    negative_returns = daily_returns[daily_returns < 0]
    metrics['Sortino Ratio'] = (daily_returns.mean() / negative_returns.std()) * np.sqrt(252) if negative_returns.std() != 0 else 0
    annual_return = daily_returns.mean() * 252
    metrics['Calmar Ratio'] = annual_return / abs(metrics['Max Drawdown (%)']/100) if metrics['Max Drawdown (%)'] != 0 else 0
    return {k: round(v, 2) for k, v in metrics.items()}

def plot_equity_curve(equity_curve, title='Strategy Performance'):
    eq_df = pd.DataFrame(equity_curve).set_index('timestamp')
    plt.style.use('seaborn-v0_8-darkgrid')
    plt.figure(figsize=(12, 6))
    plt.plot(eq_df.index, eq_df['equity'])
    plt.title(title, fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Equity ($)', fontsize=12)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gcf().autofmt_xdate()
    plt.show()

## 5. Main Analysis Function

In [ ]:
def run_full_analysis_for_instrument(ticker):
    print(f"\n{'='*80}\nRunning Full Analysis for: {ticker}\n{'='*80}")
    
    # 1. Fetch Data
    df, df_4h = fetch_data(ticker)
    if df is None: return
    
    # 2. Define base parameters
    base_params = {'Period_Fast_SMA': 20, 'Period_Medium_SMA': 50, 'Period_Slow_SMA': 200, 'ADX_Threshold': 25.0, 'ATR_Period': 14}
    
    # 3. Calculate Indicators
    df_4h.ta.sma(length=base_params['Period_Fast_SMA'], append=True)
    df_4h.ta.sma(length=base_params['Period_Medium_SMA'], append=True)
    df_4h.ta.sma(length=base_params['Period_Slow_SMA'], append=True)
    df_4h.ta.adx(length=base_params['ATR_Period'], append=True)
    df.ta.atr(length=base_params['ATR_Period'], append=True)
    df.ta.vwap(append=True)
    df.dropna(inplace=True); df_4h.dropna(inplace=True)
    
    # 4. Pre-calculate swing points
    swing_high_indices, swing_low_indices = find_swing_points(df['High'], order=10)
    df['is_swing_high'] = False; df['is_swing_low'] = False
    df.iloc[swing_high_indices, df.columns.get_loc('is_swing_high')] = True
    df.iloc[swing_low_indices, df.columns.get_loc('is_swing_low')] = True
    
    # 5. In-Sample/Out-of-Sample Split
    split_index = int(len(df) * 0.8)
    df_in_sample, df_out_of_sample = df.iloc[:split_index], df.iloc[split_index:]
    
    # 6. In-Sample Optimization
    print("\n--- Running In-Sample Parameter Optimization ---")
    param_grid = {'ATR_Multiplier': [2.0, 2.5], 'bos_atr_multiplier': [1.0, 1.5]}
    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    results = []
    for combo in param_combinations:
        current_params = {**base_params, **combo}
        bt = Backtester(df_in_sample, df_4h, current_params)
        temp_log = bt.run()
        metrics = calculate_performance_metrics(temp_log, bt.equity_curve)
        results.append({'params': combo, 'metrics': metrics})
    
    best_result = max(results, key=lambda x: x['metrics'].get('Sharpe Ratio', -np.inf))
    best_params = {**base_params, **best_result['params']}
    print("\n--- Best In-Sample Parameters Found ---")
    print(best_result['params'])
    print("\n--- Best In-Sample Performance ---")
    display(best_result['metrics'])
    
    # 7. Out-of-Sample Validation
    print("\n--- Running Out-of-Sample Validation ---")
    oos_backtester = Backtester(df_out_of_sample, df_4h, best_params)
    oos_trade_log = oos_backtester.run()
    print("\n--- Out-of-Sample Performance ---")
    oos_metrics = calculate_performance_metrics(oos_trade_log, oos_backtester.equity_curve)
    display(oos_metrics)
    plot_equity_curve(oos_backtester.equity_curve, title=f'Out-of-Sample Equity Curve for {ticker}')

## 6. Execute Analysis for All Instruments

In [ ]:
tickers_to_analyze = ['EURUSD=X', 'GBPUSD=X', 'USDJPY=X', 'AUDJPY=X', 'EURGBP=X', 'AUDUSD=X']
for ticker in tickers_to_analyze:
    run_full_analysis_for_instrument(ticker)